In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI ,OpenAIEmbeddings
from langchain.messages import HumanMessage
import os
from IPython.display import Markdown
import tokenizers

In [11]:
load_dotenv(override=True)


True

In [33]:
tokennizer= tiktoken.encoding_for_model("gpt-4o-mini")

print(tokennizer.name)

o200k_base


In [ ]:
loader = PyPDFLoader("CV_Radi_IT.pdf")
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
encoding_name=tokennizer.name, chunk_size=300, chunk_overlap=20
)


In [14]:
chunks= loader.load_and_split(splitter)


In [15]:
print(len(chunks))

3


In [16]:
print(chunks[0].metadata)


{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-18T21:10:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-18T21:10:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'CV_Radi_IT.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [17]:
print(chunks[0].page_content)


CURRICULUM VITAE
Nom : Radi
Prénom : Radi
Date de naissance : 12 Mars 1998
Adresse : Casablanca, Maroc
Téléphone : +212 6 00 00 00 00
Email : radi.dev@gmail.com
LinkedIn : linkedin.com/in/radi-dev
Profil professionnel :
Ingénieur en informatique spécialisé en développement web et applications modernes. Expérience
dans la conception, le développement et le déploiement d’applications performantes. Capacité à
travailler en équipe agile et à résoudre des problèmes complexes.
Diplômes :
Master en Génie Informatique – Université Hassan II (2022 – 2024)
Licence en Génie Logiciel – Université Hassan II (2019 – 2022)
Baccalauréat Sciences Mathématiques (2019)
Expériences professionnelles :
Développeur Full Stack – Startup Tech (Casablanca) (Janvier 2024 – Présent)
- Développement d’applications web avec React.js et Node.js
- Conception d’API REST sécurisées
- Gestion de bases de données (MongoDB, MySQL)
- Participation aux réunions Agile (Scrum)
Stagiaire Développeur Web – Société IT (Juin 2023

In [18]:
embedding_model= OpenAIEmbeddings()


In [19]:
vecror_store= Chroma.from_documents(
documents=chunks, embedding=embedding_model, 
collection_name="cv_data_collection"
)
retriever = vecror_store.as_retriever(kwargs={"k": 10})

In [20]:
@tool
def retriever_tool(query: str) -> str:
    """
   Permet de chercher des informations sur des candidats :
   -Nom, Prénom, Diplômes
   -Expériences
    -Compétences 
    """
    relevant_chunks= retriever.invoke(query)
    context_list= [d.page_content for d in relevant_chunks]
    context= ". ".join(context_list)
    return context
@tool
def get_company_infos(company_name: str):
    """Consulter des infomrationssur l'entreprise donnée"""
    return{"company_name": company_name, "domain": "IT", "turnover": 120_870_000}
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)
my_agent= create_agent(
   model=llm,
   tools=[retriever_tool, get_company_infos],
   system_prompt="Répond à la question de l'utilisateur en utilisant les tools fournis",
)


In [26]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
agent = create_agent (
    model=llm,
    tools=[retriever_tool, get_company_infos],
    system_prompt="Répond à la question de l'utilsateur en utilisant les tools fournis"
)

In [30]:
resp =agent.invoke(input={
    "messages":[
        HumanMessage("Nom, prénom , diplômes du  RADI et les informationssur son entreprise ")
    ]
})

In [23]:
print(resp['messages'][-1].content)

Voici les informations demandées :

- **Nom** : Radi
- **Prénom** : Radi
- **Diplômes** :
  - Master en Génie Informatique – Université Hassan II (2022 – 2024)
  - Licence en Génie Logiciel – Université Hassan II (2019 – 2022)
  - Baccalauréat Sciences Mathématiques (2019)


In [25]:
print(display(Markdown(resp['messages'][-1].content)))

Voici les informations demandées :

- **Nom** : Radi
- **Prénom** : Radi
- **Diplômes** :
  - Master en Génie Informatique – Université Hassan II (2022 – 2024)
  - Licence en Génie Logiciel – Université Hassan II (2019 – 2022)
  - Baccalauréat Sciences Mathématiques (2019)

None


In [31]:
print(resp['messages'][-1].content)

Voici les informations concernant RADI :

### Informations sur le candidat :
- **Nom :** Radi
- **Prénom :** Radi
- **Diplômes :**
  - Master en Génie Informatique – Université Hassan II (2022 – 2024)
  - Licence en Génie Logiciel – Université Hassan II (2019 – 2022)
  - Baccalauréat Sciences Mathématiques (2019)

### Informations sur l'entreprise RADI :
- **Domaine :** IT
- **Chiffre d'affaires :** 120,870,000

Si vous avez besoin de plus d'informations, n'hésitez pas à demander !


In [32]:
print(display(Markdown(resp['messages'][-1].content)))

Voici les informations concernant RADI :

### Informations sur le candidat :
- **Nom :** Radi
- **Prénom :** Radi
- **Diplômes :**
  - Master en Génie Informatique – Université Hassan II (2022 – 2024)
  - Licence en Génie Logiciel – Université Hassan II (2019 – 2022)
  - Baccalauréat Sciences Mathématiques (2019)

### Informations sur l'entreprise RADI :
- **Domaine :** IT
- **Chiffre d'affaires :** 120,870,000

Si vous avez besoin de plus d'informations, n'hésitez pas à demander !

None
